## Predicting Leaks
In this notebook, I will be feature engineering some variables that will help us identify each pipe's likelihood of leaking.

#### Process:
- Joined pipes.csv with train.csv on pipe_id
- Created 'Age' variable for each pipe using 'Lay date' from pipes.csv and 'Date' from train.csv
- Created 'Length' variable for each pipe using Euclidean distance
- Created target variable 'Leaked' (0 for not leaked, 1 for leaked)
- Created 'is_iron' variable to distinguish whether a pipe needs to be replaced or not


### Libraries and Reading in Data

In [64]:
import itertools
import os
from pathlib import Path
import re
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from scipy.spatial import cKDTree
from sklearn.preprocessing import OneHotEncoder
from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import StratifiedKFold

In [65]:
pipes = pd.read_csv("../data/raw/pipes.csv", parse_dates=["Lay date"])
leaks = pd.read_csv("../data/raw/train.csv", parse_dates=["Date"])

### Age in years
15 pipes have no lay date, so their age is unknown, therefore I have decided to drop these pipes.

In [66]:
df = pipes.merge(leaks, on="Pipe ID", how="left")

df["Leaked"] = df["Date"].notna().astype(int)
ref_date = leaks["Date"].max()  # end of observation period
df["Age"] = (ref_date - df["Lay date"]).dt.days / 365.25  # years

df = df.dropna(subset=["Lay date"]).reset_index(drop=True)
df['Age'].describe()

count    43024.000000
mean        68.718298
std          8.607354
min         41.007529
25%         62.704997
50%         68.807666
75%         75.236824
max         81.995893
Name: Age, dtype: float64

### Pipe length

Euclidean distance between each pipe's two endpoints, in the same units as the GPS coordinates.

In [67]:
df["Length"] = np.sqrt((df["GPS x2"] - df["GPS x1"])**2 + (df["GPS y2"] - df["GPS y1"])**2)
df['Length'].describe()

count    43024.000000
mean        17.741399
std         15.769437
min          0.170000
25%          7.986762
50%         11.312891
75%         20.677073
max         74.881445
Name: Length, dtype: float64

### Finding which pipes are required to be replaced
8 pipes have no material recorded, but after further investigation, it seems these pipes are already polyurathane, (extrapolated based on their lay dates and surface). Therefore I have decided to drop these pipes. 

In [68]:
df = df.dropna(subset=['Material'])
df['is_iron'] = df['Material'].str.contains(r'iron$', regex=True).astype("Int64")
df['is_iron'].value_counts()

is_iron
1    38440
0     4576
Name: count, dtype: Int64

### Looking at lengths of those needing replacement

In [69]:
df[df["is_iron"] == 1]["Length"].sort_values(ascending=False)


6463     74.881445
6462     74.873639
5692     74.712024
7295     74.578169
7155     74.568536
           ...    
10065     0.320156
11043     0.313209
9887      0.286531
15045     0.268328
14321     0.170000
Name: Length, Length: 38440, dtype: float64

## Next steps
1. Test different models for prediction to see which suits our data the best.
2. Do more research on time series data because I believe this is very much a time series problem. (Unfortunately, I took STAT445 instead of STAT485, so I'm not sure what methods are required with time series data)
3. Maybe look into building a random forest to try and predict likelihood of leaking to identify at risk areas to target first with our work units?

### Modifying the dataset for the model
First, I'm creating a dataset with all the variables that I want to use for prediction

In [70]:
df = df[['Material', 'Surface', 'Leaked', 'Age', 'Length']]

### Hot encoding categorical variables

In [71]:
cat_cols = ['Material', 'Surface']

train_df, test_df = train_test_split(df, test_size=0.2, random_state=42)

encoder = OneHotEncoder(handle_unknown='ignore', sparse_output=False)
encoder.fit(train_df[cat_cols])

def encode(frame):
    encoded = pd.DataFrame(
        encoder.transform(frame[cat_cols]),
        columns=encoder.get_feature_names_out(cat_cols),
        index=frame.index,
    )
    return pd.concat([frame.drop(columns=cat_cols), encoded], axis=1)

train_encoded = encode(train_df)
test_encoded = encode(test_df)

### Running RandomForestClassifier
Tuning diferent parameters for the model

In [72]:
def MSPE(Y, Y_hat):
    return np.mean((Y - Y_hat) ** 2)

rng = np.random.default_rng(440)

# Use the one-hot encoded training set (the raw df still has text columns like 'cast iron')
X = train_encoded.drop(columns="Leaked")
y = train_encoded["Leaked"]
n_features = X.shape[1]

all_mtry = [m for m in [3, 4, 5, 6, 7, 8] if m <= n_features]
all_nodesize = [1, 2, 3, 5, 8, 10, 15, 20]

all_pars = pd.DataFrame(
    [(m, s) for s, m in itertools.product(all_nodesize, all_mtry)],
    columns=["mtry", "nodesize"],
)
n_pars = len(all_pars)

M = 5

cv = StratifiedKFold(
    n_splits=M,
    shuffle=True,
    random_state=440,
)

CV_errors = np.zeros((M, n_pars))

for i in range(n_pars):
    print(f"Parameter set {i + 1} of {n_pars}")

    this_mtry = int(all_pars.loc[i, "mtry"])
    this_nodesize = int(all_pars.loc[i, "nodesize"])

    for j, (train_idx, val_idx) in enumerate(cv.split(X, y)):
        fit_rf = RandomForestClassifier(
            n_estimators=500,
            max_features=this_mtry,
            min_samples_leaf=this_nodesize,
            n_jobs=-1,
            random_state=440 + i,
        )

        fit_rf.fit(X.iloc[train_idx], y.iloc[train_idx])

        pred = fit_rf.predict(X.iloc[val_idx])

        # Misclassification error
        CV_errors[j, i] = np.mean(
            pred != y.iloc[val_idx].to_numpy()
        )

names_pars = (
    all_pars["mtry"].astype(str)
    + "-"
    + all_pars["nodesize"].astype(str)
)

CV_errors = pd.DataFrame(CV_errors, columns=names_pars)

# Select the parameter set with the lowest average misclassification error
mean_errors = CV_errors.mean(axis=0)
best_idx = int(np.argmin(mean_errors.to_numpy()))

best_params = all_pars.iloc[best_idx]

print("Best parameters:")
print(best_params)
print(f"Mean CV misclassification error: {mean_errors.iloc[best_idx]:.4f}")

Parameter set 1 of 48
Parameter set 2 of 48
Parameter set 3 of 48
Parameter set 4 of 48
Parameter set 5 of 48
Parameter set 6 of 48
Parameter set 7 of 48
Parameter set 8 of 48
Parameter set 9 of 48
Parameter set 10 of 48
Parameter set 11 of 48
Parameter set 12 of 48
Parameter set 13 of 48
Parameter set 14 of 48
Parameter set 15 of 48
Parameter set 16 of 48
Parameter set 17 of 48
Parameter set 18 of 48
Parameter set 19 of 48
Parameter set 20 of 48
Parameter set 21 of 48
Parameter set 22 of 48
Parameter set 23 of 48
Parameter set 24 of 48
Parameter set 25 of 48
Parameter set 26 of 48
Parameter set 27 of 48
Parameter set 28 of 48
Parameter set 29 of 48
Parameter set 30 of 48
Parameter set 31 of 48
Parameter set 32 of 48
Parameter set 33 of 48
Parameter set 34 of 48
Parameter set 35 of 48
Parameter set 36 of 48
Parameter set 37 of 48
Parameter set 38 of 48
Parameter set 39 of 48
Parameter set 40 of 48
Parameter set 41 of 48
Parameter set 42 of 48
Parameter set 43 of 48
Parameter set 44 of 

In [73]:
# Refit with the best parameters on the full training set and evaluate on the held-out test set
final_rf = RandomForestClassifier(
    n_estimators=500,
    max_features=int(best_params["mtry"]),
    min_samples_leaf=int(best_params["nodesize"]),
    n_jobs=-1,
    random_state=440,
)
final_rf.fit(X, y)

X_test = test_encoded.drop(columns="Leaked")
y_test = test_encoded["Leaked"]
test_pred = final_rf.predict(X_test)
print(f"Test MSPE: {MSPE(y_test, test_pred):.4f}")

pd.Series(final_rf.feature_importances_, index=X.columns).sort_values(ascending=False)

Test MSPE: 0.1869


Age                      0.761239
Length                   0.089035
Material_polyurethane    0.058748
Material_gray iron       0.034090
Material_brass           0.021893
Material_wrought iron    0.008000
Material_copper          0.007326
Material_cast iron       0.006682
Surface_water            0.003288
Surface_grassland        0.002742
Surface_farmland         0.002517
Surface_structure        0.001887
Surface_road             0.001419
Surface_swamp            0.001133
dtype: float64